# Milestone 6 - the two landmark detectors, compared

**Inputs to attach:** the WFLW dataset AND the run output holding the baseline weights (the exported `landmarks24_wing.pt`, or the wing checkpoint). `detector.weights: auto` finds a single attached weights file by itself; set it explicitly if several are attached. CPU is the right accelerator here: the timing numbers are part of the deliverable and both detectors must share the hardware.

**The gate in this milestone is the mapping check.** It runs in two parts. `verify_mediapipe_mapping.py` measures every mapped point against WFLW ground truth in percent of inter-ocular distance and, as a control, scores all 478 mesh points against each ground truth point so a better index cannot hide. Read its verdict before anything else. Then open `mediapipe_mapping_overlays.png` and go through the milestone-1 checklist by eye: pupils 12/13 dead centre (crosshairs), eyelids 0-5/6-11 tracing the lids in order, mouth 14-17 on corners and outer-lip midpoints, nose tip and chin on the axis, contour points symmetric. The two detectors must emit identical semantics or the ablation compares nothing.

Then the report: per-detector detection rates (target face = largest annotated face per image), NME on matched faces, the paired comparison on jointly matched faces with the per-point offset table, the GT-box vs Haar-box price for our model, per-frame timing on this CPU, and the footprint comparison.

In [ ]:
!rm -rf dms-layer1
!git clone -b claude/facial-landmark-perception-q80yhn https://github.com/keerthanpragnay1728-prog/dms-layer1.git
%cd dms-layer1
# provenance: confirm the commit this run uses BEFORE trusting any number
!git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
!python tests/run_tests.py

In [ ]:
# The mapping gate: MediaPipe's 24 indices against ground truth, per point.
# MediaPipe's Tasks runtime links EGL/GLES even on CPU. Kaggle images have
# both; if the wrapper reports a missing libEGL.so.1, run this first:
#   !apt-get -qq update && apt-get -qq install -y libegl1 libgles2
!python scripts/verify_mediapipe_mapping.py --config configs/layer1_base.yaml --split test --limit 300


In [ ]:
# full test split; add --limit 300 first if you want a quick pass
!python scripts/compare_detectors.py --config configs/layer1_base.yaml --split test

In [ ]:
from pathlib import Path
from IPython.display import Image, display
for name in ('mediapipe_mapping_overlays.png', 'side_by_side.png'):
    p = Path('/kaggle/working/m6_compare') / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))